# Lesson 20 - Capstone: A Validated Payroll Report

**Goal:** Combine functions, Decimal, validation, structured results, reporting, and tests in one reproducible workflow.

**Level:** intermediate. Complete the preceding chapters first. This notebook defines its own
inputs and runs independently. Run cells top to bottom with **Shift+Enter**. Predict results,
inspect the output, and change one input at a time. No external services or third-party packages
are required beyond your Jupyter setup.

Examples include lightweight executable checks. Practice cells intentionally contain no solutions;
expected results help you verify your own work. Temporary files and databases are isolated and cleaned up.

Where salary examples appear, use the course's simplified exercise rules: DA 48% of basic,
HRA 16% capped at 16,000 monthly, PF 12% of basic plus DA, and annual tax 20% only on annual
gross above 12,00,000, divided by 12 for monthly tax. PF is a separate deduction.
These are learning assumptions, not actual tax or payroll rules.

## 1. Define the input contract

Use synthetic records and explicit worksheet rules. Inputs must have a unique positive integer employee ID, a nonblank name, and finite nonnegative monetary strings. Booleans are not employee IDs even though bool is a subclass of int. Rounding uses ROUND_HALF_UP to cents for DA, HRA, PF, annual tax, and monthly tax. PF is separate from the annual gross tax threshold.

In [1]:
from dataclasses import dataclass, asdict
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
import json

CENT = Decimal("0.01")

def money(value):
    return value.quantize(CENT, rounding=ROUND_HALF_UP)

def parse_money(text):
    if not isinstance(text, str):
        raise TypeError("Money input must be text")
    try:
        value = Decimal(text.strip())
    except InvalidOperation as error:
        raise ValueError("Money must be numeric") from error
    if not value.is_finite() or value < 0:
        raise ValueError("Money must be finite and nonnegative")
    # Bounds keep the demo within a documented, realistic numeric range.
    if value > Decimal("1000000000"):
        raise ValueError("Money exceeds the worksheet input limit")
    return money(value)

assert parse_money("52500.00") == Decimal("52500.00")

## 2. Pure calculation and structured output

Calculation takes validated amounts and returns a frozen result. The worksheet assumes constant pay over 12 months and excludes real tax slabs, rebates, cess, and other payroll adjustments. Net is based on the rounded monthly deductions.

In [2]:
@dataclass(frozen=True)
class PayResult:
    gross: Decimal
    pf: Decimal
    monthly_tax: Decimal
    net: Decimal

def calculate_pay(basic, electricity):
    if not all(value.is_finite() and value >= 0 for value in (basic, electricity)):
        raise ValueError("Invalid calculation input")
    da = money(basic * Decimal("0.48"))
    hra = min(money(basic * Decimal("0.16")), Decimal("16000.00"))
    gross = basic + da + hra + electricity
    pf = money((basic + da) * Decimal("0.12"))
    annual_tax = money(max(Decimal("0"), gross * 12 - Decimal("1200000")) * Decimal("0.20"))
    monthly_tax = money(annual_tax / 12)
    return PayResult(gross, pf, monthly_tax, gross - pf - monthly_tax)

result = calculate_pay(parse_money("75000"), parse_money("1500"))
print(result)
assert result.net == Decimal("106280.00")

PayResult(gross=Decimal('124500.00'), pf=Decimal('13320.00'), monthly_tax=Decimal('4900.00'), net=Decimal('106280.00'))


## 3. Validate a batch and retain rejection reasons

Reserve an employee ID only after the record passes validation. The policy here accepts the first valid record for an ID and rejects later duplicates. A rejected invalid row does not reserve its ID. Expected input errors are collected; unrelated programming errors remain visible.

In [3]:
def process_batch(records):
    accepted, rejected = [], []
    seen = set()
    for row_number, record in enumerate(records, start=1):
        try:
            if not isinstance(record, dict):
                raise ValueError("Expected a record dictionary")
            employee_id = record["id"]
            if type(employee_id) is not int or employee_id <= 0:
                raise ValueError("ID must be a positive integer")
            if employee_id in seen:
                raise ValueError("Duplicate employee ID")
            name = record["name"]
            if not isinstance(name, str) or not name.strip():
                raise ValueError("Name must be nonblank text")
            basic = parse_money(record["basic"])
            electricity = parse_money(record.get("electricity", "1000"))
            pay = calculate_pay(basic, electricity)
            accepted.append({"id": employee_id, "name": name.strip(), "pay": pay})
            seen.add(employee_id)
        except (KeyError, TypeError, ValueError) as error:
            rejected.append({"row": row_number, "reason": str(error)})
    return accepted, rejected

records = [
    {"id": 101, "name": " Asha ", "basic": "52500"},
    {"id": 102, "name": "Manoj", "basic": "112600"},
    {"id": 101, "name": "Duplicate", "basic": "60000"},
    {"id": 103, "name": "Invalid", "basic": "nan"},
]
accepted, rejected = process_batch(records)
print([(row["id"], row["pay"].net) for row in accepted])
print(rejected)
assert len(accepted) == 2 and len(rejected) == 2

[(101, Decimal('77776.00')), (102, Decimal('146920.64'))]
[{'row': 3, 'reason': 'Duplicate employee ID'}, {'row': 4, 'reason': 'Money must be finite and nonnegative'}]


## 4. Serialize and verify the report

Convert Decimal fields explicitly to strings. Sort records by ID for stable reports. Write to a temporary folder and read back to verify the artifact. This example does not silently overwrite a real payroll file.

In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory

report_rows = [{"id": row["id"], "name": row["name"],
                **{key: f"{value:.2f}" for key, value in asdict(row["pay"]).items()}}
               for row in sorted(accepted, key=lambda item: item["id"])]
total_net = sum((row["pay"].net for row in accepted), Decimal("0.00"))
report = {"employees": report_rows, "rejected": rejected,
          "total_monthly_net": f"{total_net:.2f}"}
with TemporaryDirectory() as folder:
    path = Path(folder) / "payroll_report.json"
    path.write_text(json.dumps(report, indent=2, allow_nan=False), encoding="utf-8")
    loaded = json.loads(path.read_text(encoding="utf-8"))
assert loaded == report
assert total_net == Decimal("224696.64")
print(json.dumps(loaded, indent=2))

{
  "employees": [
    {
      "id": 101,
      "name": "Asha",
      "gross": "87100.00",
      "pf": "9324.00",
      "monthly_tax": "0.00",
      "net": "77776.00"
    },
    {
      "id": 102,
      "name": "Manoj",
      "gross": "183648.00",
      "pf": "19997.76",
      "monthly_tax": "16729.60",
      "net": "146920.64"
    }
  ],
  "rejected": [
    {
      "row": 3,
      "reason": "Duplicate employee ID"
    },
    {
      "row": 4,
      "reason": "Money must be finite and nonnegative"
    }
  ],
  "total_monthly_net": "224696.64"
}


## 5. Boundary and failure checks

Verify known results, missing fields, duplicate IDs, invalid money, zero values, and empty batches. These small checks exercise behavior rather than just duplicating the formula.

In [5]:
for basic, expected_net in [("52500", "77776.00"),
                            ("112600", "146920.64"),
                            ("120000", "154368.00")]:
    pay = calculate_pay(parse_money(basic), parse_money("1000"))
    assert pay.net == Decimal(expected_net)
assert process_batch([]) == ([], [])
zero = calculate_pay(Decimal("0"), Decimal("0"))
assert zero.net == 0
for bad in ["", "-1", "nan", "inf", "unknown"]:
    try:
        parse_money(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Expected rejection: {bad!r}")
bad_records = [{"id": True, "name": "A", "basic": "1"},
               {"id": 1, "name": " ", "basic": "1"},
               {"id": 2, "name": "B"}]
assert len(process_batch(bad_records)[1]) == 3
print("Known salaries, invalid records, zero, and empty batch checks passed.")

Known salaries, invalid records, zero, and empty batch checks passed.


## Practice lab

Define all inputs in each answer cell, use `practice_` names where helpful, and explain important
decisions in comments. Try a task before looking at its expected results. Add at least one normal,
one boundary, and one invalid-input check where the task has an input contract.
Keep file exercises inside TemporaryDirectory until you deliberately choose a permanent output path.

### Exercise 1. Add department

Require a nonblank department field, normalize it, and summarize accepted net pay by department.

**Check:** Department totals reconcile exactly with overall Decimal total.

In [6]:
# Write your solution and checks here.

### Exercise 2. Read CSV input

Load records from CSV; convert IDs with validation before processing.

**Check:** A bad ID becomes a row rejection, not a process-wide crash.

In [7]:
# Write your solution and checks here.

### Exercise 3. Add cap and threshold tests

Write unittest cases for HRA at basic 100,000 and annual gross exactly 1,200,000.

**Check:** HRA 16,000 and annual tax zero at those boundaries.

In [8]:
# Write your solution and checks here.

### Exercise 4. Persist report rows

Store accepted report results in temporary SQLite with unique employee IDs and integer cents.

**Check:** Total stored net cents equals the JSON report total multiplied by 100.

In [9]:
# Write your solution and checks here.

### Exercise 5. Extract a package

Split parsing, calculations, reporting, and tests into modules in a temporary project.

**Check:** Importing a module does not generate a report or run the whole batch.

In [10]:
# Write your solution and checks here.

## Chapter project

Complete a standalone payroll workflow: CSV input, validated records, Decimal calculation, JSON report, optional SQLite storage, and unittest verification. Document the worksheet formulas, rounding policy, duplicate-ID handling, and rejected-row policy. Use only synthetic data.

In [11]:
# Build your chapter project here. Separate inputs, logic, and display.

## Reflection

1. Which input assumptions does your solution enforce, and which does it document?
2. What happens for empty input, a boundary value, or a malformed record?
3. Which parts can be reused without the notebook?
4. What would make the code clearer to another learner?

Restart the kernel and run all cells after completing your work to catch hidden dependencies.

**Next:** `21_mysql_with_python.ipynb` - persist employee records using MySQL and integrate them with the capstone.

**Further study:** packaging, profiling, HTTP APIs, concurrency, and a project of your own.